In [41]:
!pip install scikit-learn pandas numpy faiss-cpu -q rapidfuzz
print("Libraries installed!")

Libraries installed!


## Step 1 : Load the Master Dataset

We load `master_dataset_lean.csv`, our cleaned and enriched dataset of
150,288 unique perfumes. This was assembled from three Kaggle sources
and processed through our data pipeline notebook.

The dataset contains notes, accords, ratings, olfactive families,
flanker relationships, occasion tags, and popularity scores - everything
the recommendation engine needs.

In [42]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import re
import json
import warnings
warnings.filterwarnings('ignore')

print("Loading master dataset...")
df = pd.read_csv('master_dataset_lean.csv')
df = df.reset_index(drop=True)

print(f"Loaded {len(df):,} perfumes")
print(f"Columns: {df.columns.tolist()}")

print(f"\nMissing value summary:")
missing     = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_summary = pd.DataFrame({
    'missing_count': missing,
    'missing_pct':   missing_pct
})
print(missing_summary[missing_summary['missing_count'] > 0].to_string())

# Flag perfumes with no usable note data
df['has_notes'] = df['all_notes'].notna() & (
    df['all_notes'].str.strip() != ''
)
print(f"\nPerfumes with note data:    {df['has_notes'].sum():,}")
print(f"Perfumes without note data: {(~df['has_notes']).sum():,}")
print("Perfumes without notes will be excluded from recommendations")
print("but kept in dataset for display/search purposes")

Loading master dataset...
Loaded 150,288 perfumes
Columns: ['id', 'name', 'brand', 'year', 'gender', 'top_notes', 'middle_notes', 'base_notes', 'all_notes', 'accords', 'rating_avg', 'rating_count', 'longevity_avg', 'sillage_avg', 'price_tier', 'season_spring', 'season_summer', 'season_autumn', 'season_winter', 'description', 'parent_perfume', 'flanker_type', 'flanker_group', 'is_flanker', 'source', 'image_url']

Missing value summary:
                missing_count  missing_pct
year                    42491         28.3
top_notes               44675         29.7
middle_notes            44781         29.8
base_notes              44712         29.8
all_notes                8313          5.5
accords                  3180          2.1
rating_avg               1281          0.9
rating_count             2308          1.5
longevity_avg           18358         12.2
sillage_avg             18358         12.2
season_spring           18358         12.2
season_summer           18358         12.2
se

## Step 2 : Fix Encoding and Standardise Text

### The Problem
Our dataset was assembled from multiple sources with different character
encodings. French, Spanish, Portuguese and Arabic perfume names contain
accented characters that got corrupted during the latin-1 read - for
example `Ãsika` instead of `Ásika`, or `VanillÃ©e` instead of `Vanillée`.

Additionally, perfume databases use inconsistent note naming conventions.
`Bergamotte`, `Bergamot`, and `bergamot` all refer to the same ingredient
but would be treated as three different terms by the TF-IDF vectorizer,
weakening the similarity matching.

### What We Do
Three layers of cleaning in sequence:

1. **Encoding repair** - fixes mojibake from latin-1/utf-8 mismatches
2. **Hardcoded variants** - corrects a curated list of known misspellings
   and multilingual variants
3. **Frequency-based autocorrection** - automatically finds rare terms
   that are likely misspellings of common terms and corrects them

In [43]:
#Cell 2b : Encoding and Hardcoded Fixes

import unicodedata

def fix_encoding(text):
    """
    Repairs mojibake - characters corrupted by latin-1/utf-8 mismatch.
    Example: 'Ãsika' → 'Ásika'
    """
    if not text or pd.isna(text):
        return text
    try:
        fixed = str(text).encode('latin-1').decode('utf-8')
        return fixed
    except (UnicodeDecodeError, UnicodeEncodeError):
        try:
            return unicodedata.normalize('NFC', str(text))
        except:
            return str(text)

def standardise_text(text):
    """
    Standardises punctuation and whitespace.
    Replaces smart quotes, em dashes, and extra spaces.
    """
    if not text or pd.isna(text):
        return text
    t = str(text)
    t = t.replace('\u2019', "'").replace('\u2018', "'")
    t = t.replace('\u2013', '-').replace('\u2014', '-')
    t = re.sub(r'\s+', ' ', t).strip()
    return t

def fix_spelling_variants(text):
    """
    Corrects known note spelling variants and multilingual equivalents.
    These are cases we identified manually from fragrance databases.
    Only corrects exact full-term matches to avoid partial replacements.
    """
    if not text or pd.isna(text):
        return text

    # Term-level replacements only - split first to avoid
    # replacing substrings inside longer terms
    variants = {
        'bergamotte':     'bergamot',
        'sandal wood':    'sandalwood',
        'sandal-wood':    'sandalwood',
        'aoud':           'oud',
        'musc':           'musk',
        'musque':         'musk',
        'musqué':         'musk',
        'vanille':        'vanilla',
        'vétiver':        'vetiver',
        'vetyver':        'vetiver',
        'vétiver':        'vetiver',
        'patchoulli':     'patchouli',
        'pachouli':       'patchouli',
        'ylang ylang':    'ylang-ylang',
        'ambre':          'amber',
        'labdanum':       'labdanum',
        'cistus':         'labdanum',
        'iso e super':    'iso-e-super',
        'tonka':          'tonka bean',
    }

    # Remove data artefacts - percentage values mixed into notes
    # e.g. 'woody100', 'citrus100', 'floral85'
    text_clean = re.sub(r'\b([a-zA-Z]+)\d+\b', r'\1', str(text))
    text_clean = re.sub(r'\b\d+([a-zA-Z]+)\b', r'\1', text_clean)

    parts     = re.split(r'([,|;\n])', text_clean)
    corrected = []
    for part in parts:
        stripped = part.strip().lower()
        if stripped in [',', '|', ';', '\n', '']:
            corrected.append(part)
        elif stripped in variants:
            corrected.append(variants[stripped])
        else:
            corrected.append(part)

    return ''.join(corrected)

# Apply to all text columns
print("Fixing encoding issues...")
text_columns = ['name', 'brand', 'description', 'all_notes',
                'top_notes', 'middle_notes', 'base_notes', 'accords']

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(fix_encoding)
        df[col] = df[col].apply(standardise_text)

note_columns = ['all_notes', 'top_notes', 'middle_notes',
                'base_notes', 'accords']
for col in note_columns:
    if col in df.columns:
        df[col] = df[col].apply(fix_spelling_variants)

print("Encoding and spelling standardisation complete!")

remaining = df['name'].str.contains(
    r'Ã|â€|Ã©|Ã¨|Ã ', na=False, regex=True
).sum()
print(f"Remaining encoding issues in names: {remaining}")

Fixing encoding issues...
Encoding and spelling standardisation complete!
Remaining encoding issues in names: 92


## Step 3 : Frequency-Based Autocorrection

### Why a Static List Is Not Enough
Our hardcoded list covers known variants but the dataset contains
23,000+ unique note terms from global perfume databases. No manually
maintained list can cover all of them.

### The Approach
We let the data itself tell us what the correct spellings are.

The core insight: if a term appears 4 times and a very similar term
appears 9,000 times, the rare one is almost certainly a misspelling
or variant of the common one.

We extract all note terms, count their frequencies, then use fuzzy
string matching to find rare terms that are likely variants of common
ones. Only matches above 88% similarity are corrected, and we add
safety checks to prevent false corrections like `sage` → `rose`.

In [44]:
# Cell 2c : Extract Term Frequencies
# ─────────────────────────────────────────────────────────────────────
# Before we can automatically detect and fix spelling variants,
# we need to know how often every note term appears across the dataset.
#
# The core insight: if a term appears 4 times and a very similar term
# appears 9,000 times, the rare one is almost certainly a misspelling.
# We let the data tell us what the correct spellings are rather than
# maintaining an ever-growing manual list.
# ─────────────────────────────────────────────────────────────────────

from collections import Counter

def extract_all_note_terms(df):
    """
    Extracts every individual note term across the full dataset
    and counts frequency of appearance.
    Splits on commas, pipes, semicolons and newlines.
    """
    all_terms = []
    for col in ['all_notes', 'top_notes', 'middle_notes',
                'base_notes', 'accords']:
        if col not in df.columns:
            continue
        for val in df[col].dropna():
            terms = re.split(r'[,|;\n]', str(val))
            for term in terms:
                cleaned = term.strip().lower()
                cleaned = re.sub(r'[^\w\s\-]', '', cleaned).strip()
                if cleaned and len(cleaned) > 2:
                    all_terms.append(cleaned)
    return Counter(all_terms)

# Extract from the already-cleaned dataframe
# (Cell 2b must have run before this)
print("Extracting note terms from cleaned dataset...")
term_frequency = extract_all_note_terms(df)

print(f"Unique note terms found: {len(term_frequency):,}")

# Check artefacts are gone after Cell 2b cleaning
print(f"\nArtefact check:")
print(f"  woody100:  {term_frequency.get('woody100',  0)} occurrences")
print(f"  citrus100: {term_frequency.get('citrus100', 0)} occurrences")
print(f"  woody:     {term_frequency.get('woody',     0):,} occurrences")
print(f"  citrus:    {term_frequency.get('citrus',    0):,} occurrences")

print(f"\nTop 30 most common note terms:")
for term, count in term_frequency.most_common(30):
    print(f"  {term:<35} {count:>8,}")

Extracting note terms from cleaned dataset...
Unique note terms found: 23,418

Artefact check:
  woody100:  19825 occurrences
  citrus100: 17736 occurrences
  woody:     11,706 occurrences
  citrus:    11,405 occurrences

Top 30 most common note terms:
  musk                                  95,592
  vanilla                               77,757
  amber                                 77,081
  bergamot                              71,296
  sandalwood                            67,075
  patchouli                             66,096
  jasmine                               65,806
  rose                                  58,040
  cedar                                 44,120
  vetiver                               35,130
  tonka bean                            30,750
  lemon                                 28,852
  lavender                              27,107
  mandarin orange                       23,081
  orange blossom                        22,664
  leather                               21

In [45]:
# Cell 2d : Build Autocorrect Map, Remove Bad Corrections,
#            Restore Legitimate Ones
# ─────────────────────────────────────────────────────────────────────
# Three stages in one cell:
#
# Stage 1 — Build: use fuzzy matching to find rare terms that are
#   likely misspellings of common terms. Safety checks prevent false
#   corrections like 'sage' → 'rose'.
#
# Stage 2 — Fix: remove corrections that are factually wrong.
#   These are cases where the fuzzy matcher matched on a shared prefix
#   but the second half of the compound phrase is a completely different
#   ingredient (e.g. 'orange blossom and basil' ≠ 'orange blossom and
#   jasmine').
#
# Stage 3 — Restore: put back corrections that the compound mismatch
#   filter incorrectly removed. These are genuine variants like
#   'vanilla pod' → 'vanilla' or 'patchouli leaf' → 'patchouli'
#   where the second word is just a descriptor, not a different note.
# ─────────────────────────────────────────────────────────────────────

from rapidfuzz import fuzz, process

# ── STAGE 1: BUILD ────────────────────────────────────────────────────

# Terms that should NEVER be corrected — distinct ingredients that
# share string similarity with other distinct ingredients
PROTECTED_TERMS = {
    'sage', 'rice', 'rose', 'pine', 'lime', 'mace', 'musk',
    'iris', 'orris', 'basil', 'cedar', 'amber', 'anise',
    'violet', 'clove', 'smoke', 'incense', 'leather', 'vanilla'
}

def build_autocorrect_map(term_frequency, similarity_threshold=88):
    """
    Identifies rare terms that are likely misspellings of common terms.

    Safety checks applied in order:
    1. Skip protected terms - distinct ingredients with similar names
    2. Skip if similarity score below threshold (default 88%)
    3. Skip if first letters differ - catches sage/rose, basil/jasmine
    4. Skip if length ratio too extreme - prevents abbreviations
       mapping to full words
    5. Skip if canonical is itself a protected term
    """
    common_terms = [
        term for term, count in term_frequency.items()
        if count >= 50
    ]
    rare_terms = [
        term for term, count in term_frequency.items()
        if count < 10 and len(term) > 3
    ]

    print(f"Common terms (50+ appearances): {len(common_terms):,}")
    print(f"Rare terms (<10 appearances):   {len(rare_terms):,}")
    print(f"Building autocorrect map...")

    autocorrect_map      = {}
    skipped_protected    = 0
    skipped_first_letter = 0
    skipped_length       = 0

    for rare in rare_terms:
        if rare in PROTECTED_TERMS:
            skipped_protected += 1
            continue

        match = process.extractOne(
            rare, common_terms, scorer=fuzz.ratio
        )

        if not match or match[1] < similarity_threshold:
            continue

        canonical = match[0]

        if rare[0] != canonical[0]:
            skipped_first_letter += 1
            continue

        if len(rare) < len(canonical) * 0.7:
            skipped_length += 1
            continue

        if canonical in PROTECTED_TERMS:
            skipped_protected += 1
            continue

        autocorrect_map[rare] = canonical

    print(f"\nAutocorrect map built: {len(autocorrect_map):,} corrections")
    print(f"Skipped (protected terms):    {skipped_protected:,}")
    print(f"Skipped (first letter check): {skipped_first_letter:,}")
    print(f"Skipped (length check):       {skipped_length:,}")

    print(f"\nSample corrections (first 15):")
    for wrong, right in list(autocorrect_map.items())[:15]:
        orig_count = term_frequency[wrong]
        can_count  = term_frequency[right]
        print(f"  '{wrong}' ({orig_count}x) → '{right}' ({can_count:,}x)")

    return autocorrect_map

autocorrect_map = build_autocorrect_map(term_frequency)

# ── STAGE 2: FIX - Remove factually wrong corrections ─────────────────

print("\n── Stage 2: Removing bad corrections ──")

# These are compound phrases where the fuzzy matcher correctly matched
# the first half but got the second half completely wrong.
# Each of these would corrupt real ingredient data if applied.
bad_corrections = [
    'orange blossom and orris',
    'orange blossom and basil',
    'mandarin orange and black pepper',
    'mandarin orange and grapes',
    'pecan and jasmine',
]

removed_bad = 0
for bad in bad_corrections:
    if bad in autocorrect_map:
        removed = autocorrect_map.pop(bad)
        print(f"  Removed: '{bad}' → '{removed}'")
        removed_bad += 1

print(f"Removed {removed_bad} known bad corrections")

# Also remove any compound phrase where the last words are too different
# This catches cases like 'orange blossom and honey' → 'orange blossom
# and rose' where the shared prefix fools the matcher
to_remove = []
for rare, canonical in autocorrect_map.items():
    rare_words      = rare.split()
    canonical_words = canonical.split()

    if len(rare_words) > 1 and len(canonical_words) > 1:
        last_rare      = rare_words[-1]
        last_canonical = canonical_words[-1]
        last_similarity = fuzz.ratio(last_rare, last_canonical)

        if last_similarity < 60:
            to_remove.append(rare)

removed_compound = 0
for rare in to_remove:
    autocorrect_map.pop(rare)
    removed_compound += 1

print(f"Removed {removed_compound} compound phrase mismatches")
print(f"Map after fixes: {len(autocorrect_map):,} corrections")

# ── STAGE 3: RESTORE - Put back legitimate corrections ─────────────────

print("\n── Stage 3: Restoring legitimate corrections ──")

# These were caught by the compound filter but are genuine variants.
# 'X leaf', 'X pod', 'X bean' are descriptors, not different ingredients.
# 'lily of the valley' vs 'lily-of-the-valley' is just hyphenation.
legitimate_corrections = {
    # Leaf descriptor variants
    'cedar and patchouli leaf':           'cedar and patchouli',
    'vanilla and patchouli leaf':         'vanilla and patchouli',
    'bergamot and black currant leaf':    'bergamot and black currant',
    'mandarin orange and lemon zest':     'mandarin orange and lemon',
    'mandarin orange and orange peel':    'mandarin orange and orange',

    # Pod/bean descriptor variants
    'white musk and vanilla pod':         'white musk and vanilla',
    'amber and vanilla pod':              'amber and vanilla',
    'patchouli and vanilla bean':         'patchouli and vanilla',
    'tonka bean and vanilla pod':         'tonka bean and vanilla',

    # Spacing and hyphenation variants — same ingredient
    'jasmine and lily of the valley':     'jasmine and lily-of-the-valley',
    'freesia and lily of the valley':     'freesia and lily-of-the-valley',

    # Tonka always means tonka bean
    'sandalwood and tonka':               'sandalwood and tonka bean',

    # Zest/peel are just parts of the fruit — same scent profile
    'bergamot and lemon zest':            'bergamot and lemon',
    'amber and lemon zest':               'amber and lemon',
    'cedar and lemon zest':               'cedar and lemon',
}

restored = 0
for wrong, right in legitimate_corrections.items():
    if wrong not in autocorrect_map:
        autocorrect_map[wrong] = right
        restored += 1

print(f"Restored {restored} legitimate corrections")
print(f"\nFinal autocorrect map: {len(autocorrect_map):,} corrections")

Common terms (50+ appearances): 4,079
Rare terms (<10 appearances):   15,630
Building autocorrect map...

Autocorrect map built: 881 corrections
Skipped (protected terms):    0
Skipped (first letter check): 216
Skipped (length check):       0

Sample corrections (first 15):
  'lillac' (6x) → 'lilac' (2,701x)
  'narciussus' (2x) → 'narcissus' (2,702x)
  'sweet pie' (7x) → 'sweet pea' (541x)
  'cashmir wood' (6x) → 'cashmirwood' (151x)
  'sea shells' (2x) → 'seashells' (62x)
  'cereal' (1x) → 'cereals' (61x)
  'orange blossom and orris' (4x) → 'orange blossom and rose' (56x)
  'sandalowood and patchouli' (4x) → 'sandalwood and patchouli' (191x)
  'vanilla and sandalowood' (4x) → 'vanilla and sandalwood' (207x)
  'orange blossom and basil' (2x) → 'orange blossom and jasmine' (132x)
  'musk and oak moss' (8x) → 'musk and oakmoss' (80x)
  'mandarin orange and black pepper' (8x) → 'mandarin orange and pink pepper' (67x)
  'pecan and jasmine' (2x) → 'peony and jasmine' (87x)
  'mandarin orang

In [46]:
# Cell 2e — Apply Autocorrections to All Note Columns
# ─────────────────────────────────────────────────────────────────────
# We reload from the saved CSV and re-apply ALL cleaning steps in the
# correct order. This ensures:
#   1. No corrections are double-applied
#   2. The final df reflects every cleaning decision made so far
#   3. The order is always: encode fix → spelling variants → autocorrect
#
# Cleaning order matters. Encoding must come first because a corrupted
# character in a note name ('Ã©' instead of 'é') would cause the term
# to not match anything in our autocorrect map.
# ─────────────────────────────────────────────────────────────────────

def apply_autocorrect(text, autocorrect_map):
    """
    Applies autocorrections term by term.
    Splits on separators first so we never accidentally correct
    a substring within a longer compound note string.
    """
    if not text or pd.isna(text):
        return text
    parts     = re.split(r'([,|;\n])', str(text))
    corrected = []
    for part in parts:
        stripped = part.strip().lower()
        if stripped in [',', '|', ';', '\n', '']:
            corrected.append(part)
        elif stripped in autocorrect_map:
            corrected.append(autocorrect_map[stripped])
        else:
            corrected.append(part)
    return ''.join(corrected)

print("Reloading dataset and applying all cleaning from scratch...")
df = pd.read_csv('master_dataset_lean.csv')
df = df.reset_index(drop=True)

text_columns = ['name', 'brand', 'description', 'all_notes',
                'top_notes', 'middle_notes', 'base_notes', 'accords']
note_columns = ['all_notes', 'top_notes', 'middle_notes',
                'base_notes', 'accords']

# Step 1: Encoding fix + punctuation standardisation (all text columns)
print("Step 1: Fixing encoding and punctuation...")
for col in text_columns:
    if col in df.columns:
        df[col] = df[col].apply(fix_encoding)
        df[col] = df[col].apply(standardise_text)

# Step 2: Hardcoded spelling variants + artefact removal (note columns)
print("Step 2: Applying hardcoded spelling variants...")
for col in note_columns:
    if col in df.columns:
        df[col] = df[col].apply(fix_spelling_variants)

# Step 3: Frequency-based autocorrections (note columns)
print("Step 3: Applying frequency-based autocorrections...")
corrections_made = 0
for col in note_columns:
    if col not in df.columns:
        continue
    original = df[col].copy()
    df[col]  = df[col].apply(
        lambda x: apply_autocorrect(x, autocorrect_map)
    )
    changed         = (original != df[col]).sum()
    corrections_made += changed
    print(f"  {col}: {changed:,} cells updated")

# Step 4: Re-add the has_notes flag
df['has_notes'] = df['all_notes'].notna() & (
    df['all_notes'].str.strip() != ''
)

print(f"\nTotal corrections applied: {corrections_made:,}")

# ── Final vocabulary verification ─────────────────────────────────────
print("\nVerifying final vocabulary...")
final_freq = extract_all_note_terms(df)

print(f"\nVocabulary size: {len(final_freq):,} unique terms")
print(f"\nArtefact check:")
print(f"  woody100:  {final_freq.get('woody100',  0)} occurrences")
print(f"  citrus100: {final_freq.get('citrus100', 0)} occurrences")
print(f"  woody:     {final_freq.get('woody',     0):,} occurrences")
print(f"  citrus:    {final_freq.get('citrus',    0):,} occurrences")

print(f"\nFinal top 15 note terms:")
for term, count in final_freq.most_common(15):
    print(f"  {term:<35} {count:>8,}")

print(f"\nDataset summary:")
print(f"  Total perfumes:      {len(df):,}")
print(f"  With note data:      {df['has_notes'].sum():,}")
print(f"  Without note data:   {(~df['has_notes']).sum():,}")
print(f"\nDataset is clean and ready for feature engineering!")

Reloading dataset and applying all cleaning from scratch...
Step 1: Fixing encoding and punctuation...
Step 2: Applying hardcoded spelling variants...
Step 3: Applying frequency-based autocorrections...
  all_notes: 8,625 cells updated
  top_notes: 44,739 cells updated
  middle_notes: 44,854 cells updated
  base_notes: 44,896 cells updated
  accords: 3,184 cells updated

Total corrections applied: 146,298

Verifying final vocabulary...

Vocabulary size: 23,241 unique terms

Artefact check:
  woody100:  19825 occurrences
  citrus100: 17736 occurrences
  woody:     11,706 occurrences
  citrus:    11,405 occurrences

Final top 15 note terms:
  musk                                  95,592
  vanilla                               77,757
  amber                                 77,081
  bergamot                              71,296
  sandalwood                            67,075
  patchouli                             66,096
  jasmine                               65,806
  rose                  

In [47]:
# Cell 2f : Verify Final Vocabulary
# ─────────────────────────────────────────────────────────────────────
# Cell 2c ran on uncleaned data so its term_frequency Counter is stale.
# We re-extract here from the fully cleaned dataframe to get accurate
# vocabulary counts before moving into feature engineering.
# ─────────────────────────────────────────────────────────────────────

print("Re-extracting term frequency from fully cleaned dataframe...")
term_frequency = extract_all_note_terms(df)

print(f"Vocabulary size: {len(term_frequency):,} unique terms")

print(f"\nArtefact check:")
print(f"  woody100:  {term_frequency.get('woody100',  0)} occurrences")
print(f"  citrus100: {term_frequency.get('citrus100', 0)} occurrences")
print(f"  woody:     {term_frequency.get('woody',     0):,} occurrences")
print(f"  citrus:    {term_frequency.get('citrus',    0):,} occurrences")

print(f"\nTop 15 terms after full cleaning pipeline:")
for term, count in term_frequency.most_common(15):
    print(f"  {term:<35} {count:>8,}")

print(f"\nDataset is confirmed clean and ready for Cell 3!")

Re-extracting term frequency from fully cleaned dataframe...
Vocabulary size: 23,241 unique terms

Artefact check:
  woody100:  19825 occurrences
  citrus100: 17736 occurrences
  woody:     11,706 occurrences
  citrus:    11,405 occurrences

Top 15 terms after full cleaning pipeline:
  musk                                  95,592
  vanilla                               77,757
  amber                                 77,081
  bergamot                              71,296
  sandalwood                            67,075
  patchouli                             66,096
  jasmine                               65,806
  rose                                  58,040
  cedar                                 44,120
  vetiver                               35,130
  tonka bean                            30,750
  lemon                                 28,852
  lavender                              27,107
  mandarin orange                       23,081
  orange blossom                        22,664

Dataset i

In [48]:
# Cell 2g : Fix accord parsing and extract accord strengths
# ─────────────────────────────────────────────────────────────────────
# The accords column stores data as "accord_name:percentage" pairs
# separated by pipes. Example:
#   'citrus:100|woody:56|warm spicy:51|amber:40'
#
# Previously our extraction was stripping the colon and producing
# artefacts like 'citrus100'. We now parse these properly:
#   1. Extract clean accord names (without percentages) for TF-IDF
#   2. Extract accord:strength pairs for a new weighted accords column
#   3. Fix extract_all_note_terms to handle the colon format
# ─────────────────────────────────────────────────────────────────────

def parse_accords_column(accord_string):
    """
    Parses accord strings that may contain percentage scores.

    Handles both formats:
      Format A: 'citrus:100|woody:56|amber:40'  (with scores)
      Format B: 'citrus, woody, amber'           (plain text)

    Returns:
      clean_accords  - comma separated accord names only
      accord_weights - dict of {accord: strength_percentage}
    """
    if not accord_string or pd.isna(accord_string):
        return None, {}

    accord_string = str(accord_string).strip()
    accord_weights = {}
    clean_names    = []

    # Detect format A - contains colon+number pattern
    if re.search(r'\w+:\d+', accord_string):
        parts = accord_string.split('|')
        for part in parts:
            part = part.strip()
            match = re.match(r'^(.+?):(\d+)$', part)
            if match:
                name     = match.group(1).strip().lower()
                strength = int(match.group(2))
                clean_names.append(name)
                accord_weights[name] = strength
            elif part:
                clean_names.append(part.lower().strip())

    # Format B - plain comma or pipe separated
    else:
        separators = r'[,|;]'
        parts      = re.split(separators, accord_string)
        for part in parts:
            name = part.strip().lower()
            if name:
                clean_names.append(name)
                accord_weights[name] = 100  # equal weight if no scores

    clean = ', '.join(n for n in clean_names if n)
    return clean if clean else None, accord_weights

def build_weighted_accord_string(accord_weights, top_n=5):
    """
    Builds a weighted accord string for TF-IDF by repeating
    high-strength accords more times than weak ones.

    Strength tiers:
      80-100 → repeated 3 times (dominant accord)
      50-79  → repeated 2 times (strong accord)
      1-49   → repeated 1 time  (supporting accord)
    """
    if not accord_weights:
        return None

    weighted_parts = []
    sorted_accords = sorted(
        accord_weights.items(),
        key=lambda x: x[1],
        reverse=True
    )[:top_n]

    for accord, strength in sorted_accords:
        if strength >= 80:
            weighted_parts.extend([accord] * 3)
        elif strength >= 50:
            weighted_parts.extend([accord] * 2)
        else:
            weighted_parts.append(accord)

    return ' '.join(weighted_parts)

# Apply to dataset
print("Parsing accord columns...")
parsed_accords   = df['accords'].apply(parse_accords_column)
df['accords']    = parsed_accords.apply(lambda x: x[0])
df['accord_weights'] = parsed_accords.apply(lambda x: x[1])
df['accords_weighted'] = df['accord_weights'].apply(
    build_weighted_accord_string
)

# Verify
sample = df[df['accords'].notna()].iloc[0]
print(f"\nSample accord parsing:")
print(f"  Name:             {sample['name']}")
print(f"  Clean accords:    {sample['accords']}")
print(f"  Accord weights:   {sample['accord_weights']}")
print(f"  Weighted string:  {sample['accords_weighted']}")

# Fix extract_all_note_terms to handle clean accords only
def extract_all_note_terms(df):
    """
    Updated version - uses cleaned accords column (no percentage scores)
    so woody100 and citrus100 artefacts can never appear.
    """
    all_terms = []

    # Note columns — split on separators
    for col in ['all_notes', 'top_notes', 'middle_notes', 'base_notes']:
        if col not in df.columns:
            continue
        for val in df[col].dropna():
            terms = re.split(r'[,|;\n]', str(val))
            for term in terms:
                cleaned = term.strip().lower()
                cleaned = re.sub(r'[^\w\s\-]', '', cleaned).strip()
                if cleaned and len(cleaned) > 2:
                    all_terms.append(cleaned)

    # Accords — now clean, no percentages, split on commas only
    if 'accords' in df.columns:
        for val in df['accords'].dropna():
            terms = re.split(r'[,]', str(val))
            for term in terms:
                cleaned = term.strip().lower()
                cleaned = re.sub(r'[^\w\s\-]', '', cleaned).strip()
                if cleaned and len(cleaned) > 2:
                    all_terms.append(cleaned)

    return Counter(all_terms)

# Re-extract with fixed function
print("\nRe-extracting term frequency with fixed function...")
term_frequency = extract_all_note_terms(df)

print(f"Vocabulary size: {len(term_frequency):,} unique terms")
print(f"\nArtefact check:")
print(f"  woody100:  {term_frequency.get('woody100',  0)} occurrences")
print(f"  citrus100: {term_frequency.get('citrus100', 0)} occurrences")
print(f"  woody:     {term_frequency.get('woody',     0):,} occurrences")
print(f"  citrus:    {term_frequency.get('citrus',    0):,} occurrences")

print(f"\nTop 15 terms:")
for term, count in term_frequency.most_common(15):
    print(f"  {term:<35} {count:>8,}")

# How many perfumes have accord strength data
has_weights = df['accord_weights'].apply(lambda x: len(x) > 0).sum()
print(f"\nPerfumes with accord strength data: {has_weights:,}")
print(f"Sample weighted accord strings:")
print(
    df[df['accords_weighted'].notna()][['name', 'accords_weighted']]
    .head(5)
    .to_string(index=False)
)

Parsing accord columns...

Sample accord parsing:
  Name:             Orange Tonic
  Clean accords:    citrus, fresh spicy, green, sweet, aromatic, fruity, white floral, honey
  Accord weights:   {'citrus': 100, 'fresh spicy': 47, 'green': 41, 'sweet': 33, 'aromatic': 31, 'fruity': 27, 'white floral': 26, 'honey': 16}
  Weighted string:  citrus citrus citrus fresh spicy green sweet aromatic

Re-extracting term frequency with fixed function...
Vocabulary size: 16,616 unique terms

Artefact check:
  woody100:  0 occurrences
  citrus100: 0 occurrences
  woody:     93,138 occurrences
  citrus:    71,374 occurrences

Top 15 terms:
  amber                                122,559
  vanilla                              112,319
  musk                                  95,592
  woody                                 93,138
  rose                                  83,657
  patchouli                             80,918
  powdery                               72,146
  citrus                             

## Step 4 : Build Feature Strings

This is the most important step before the TF-IDF vectorizer.

Every perfume needs to be represented as a single rich text string
that captures its complete scent identity. The TF-IDF vectorizer
will learn from these strings and convert them into numerical vectors
that can be compared mathematically.

### Weighting Strategy
Not all fields are equally important for scent matching. We weight
fields by repeating them in the feature string:

- **Notes** - repeated 3 times (most important: the actual ingredients)
- **Weighted accords** - already weighted internally by strength score
- **Olfactive family** - repeated 2 times (broad scent category)
- **Gender, occasion, season** - repeated once each (contextual filters)
- **Flanker type** - helps group similar versions of the same perfume

This means a perfume dominated by vanilla and amber will have those
terms appear many more times in its vector than a supporting note
like lemon, which is exactly how a human nose would weight them.

In [49]:
# Cell 3 : Build Feature Strings
# ─────────────────────────────────────────────────────────────────────
# Combines all scent-relevant fields into one rich text string per
# perfume. This string is what the TF-IDF vectorizer learns from.
#
# Key design decisions:
#   - Notes repeated 3x: they are the ground truth of what a perfume
#     smells like
#   - accords_weighted already encodes strength via repetition
#   - Olfactive family repeated 2x: broad category matters a lot for
#     similarity (you wouldn't recommend a fresh citrus to someone
#     who loves heavy orientals)
#   - Punctuation stripped: TF-IDF works on clean word tokens
# ─────────────────────────────────────────────────────────────────────

def build_feature_string(row):
    """
    Builds a single text string representing a perfume's scent identity.
    Used as input to TF-IDF vectorization.
    """
    parts = []

    # Notes — most important signal, repeated 3 times
    if pd.notna(row.get('all_notes')) and str(row.get('all_notes')).strip():
        notes_clean = re.sub(r'[|]', ' ', str(row['all_notes']))
        parts += [notes_clean] * 3

    # Weighted accords — strength already encoded via repetition
    # from build_weighted_accord_string in Cell 2g
    if pd.notna(row.get('accords_weighted')) and \
       str(row.get('accords_weighted')).strip():
        parts.append(str(row['accords_weighted']))

    # Plain accords as backup if weighted version is missing
    elif pd.notna(row.get('accords')) and str(row.get('accords')).strip():
        parts += [str(row['accords'])] * 2

    # Olfactive family — broad scent category, repeated 2 times
    if pd.notna(row.get('olfactive_family')) and \
       str(row.get('olfactive_family')) not in ['Other', 'nan']:
        parts += [str(row['olfactive_family'])] * 2

    # Contextual fields — once each
    if pd.notna(row.get('gender')):
        parts.append(str(row['gender']))

    if pd.notna(row.get('occasion_tags')):
        parts.append(str(row['occasion_tags']))

    if pd.notna(row.get('best_season')):
        parts.append(str(row['best_season']))

    if pd.notna(row.get('flanker_type')) and \
       str(row.get('flanker_type')) != 'original':
        parts.append(str(row['flanker_type']))

    # Combine and clean for TF-IDF
    combined = ' '.join(parts)
    combined = re.sub(r'[^\w\s]', ' ', combined)
    combined = re.sub(r'\s+', ' ', combined).strip().lower()

    return combined if combined.strip() else None

print("Building feature strings for all perfumes...")
df['feature_string'] = df.apply(build_feature_string, axis=1)

# Validation
total          = len(df)
with_features  = df['feature_string'].notna().sum()
without_features = total - with_features

print(f"\nFeature strings built:")
print(f"  Total perfumes:         {total:,}")
print(f"  With feature string:    {with_features:,}")
print(f"  Without feature string: {without_features:,}")

# Show sample feature strings
print(f"\nSample feature strings:")
samples = df[df['feature_string'].notna()].head(3)
for _, row in samples.iterrows():
    print(f"\n  {row['name']} by {row['brand']}")
    print(f"  {str(row['feature_string'])[:200]}...")

# Average feature string length
avg_len = df['feature_string'].dropna().str.len().mean()
print(f"\nAverage feature string length: {avg_len:.0f} characters")

Building feature strings for all perfumes...

Feature strings built:
  Total perfumes:         150,288
  With feature string:    150,288
  Without feature string: 0

Sample feature strings:

  Orange Tonic by Azzaro
  orange bergamot basil grapefruit green notes mint lemon honeysuckle blueberry flowers lily of the valley jasmine honey amber cedar musk orange bergamot basil grapefruit green notes mint lemon honeysuc...

  Amarige by Givenchy
  orange blossom peach plum neroli brazilian rosewood violet mandarin orange tuberose mimosa gardenia ylang ylang jasmine black locust carnation black currant red berries rose cassia orchid sandalwood a...

  Organza by Givenchy
  nutmeg gardenia african orange flower green notes bergamot tuberose jasmine honeysuckle iris peony mace vanilla amber woodsy notes guaiac wood virginia cedar nutmeg gardenia african orange flower gree...

Average feature string length: 315 characters


## Step 5 : Load Perfumer and Similar Perfume Data

We enrich the dataset with two additional signals from `perfumes.jsonl`:

### Perfumer Data
The nose (perfumer) who created a fragrance is a strong signal for
recommendation. People who love one creation by Olivier Polge or
Alberto Morillas often love others. We extract perfumer credits and
store them for use in the affinity boost layer of the recommender.

### Similar Perfumes
Fragrantica pre-computes a "similar perfumes" list for each entry
based on actual user browsing behavior - what people look at together.
This is essentially crowd-sourced collaborative filtering and is one
of the strongest signals we have before our own users generate data.

We use these similar perfume lists as soft seeds in the recommendation
function, blended at 20% weight alongside our own TF-IDF similarity.

In [50]:
# Cell 3b — Load Perfumer Data from perfumes.jsonl
# ─────────────────────────────────────────────────────────────────────
# Perfumers are stored as a list of dicts:
# [{'name': 'Dominique Ropion', 'slug': '...', 'image_id': 1}]
# We extract just the 'name' field from each dict.
# ─────────────────────────────────────────────────────────────────────

print("Loading perfumer data from perfumes.jsonl...")
perfumer_map = {}

with open('perfumes.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        try:
            record    = json.loads(line.strip())
            name      = str(record.get('name',  '')).strip()
            brand     = str(record.get('brand', '')).strip()
            perfumers = record.get('perfumers', [])

            if not name:
                continue

            key = f"{name.lower()}|{brand.lower()}"

            # Extract name field from each perfumer dict
            if isinstance(perfumers, list) and len(perfumers) > 0:
                names = []
                for p in perfumers:
                    if isinstance(p, dict) and p.get('name'):
                        names.append(str(p['name']).strip())
                    elif isinstance(p, str) and p.strip():
                        names.append(p.strip())
                if names:
                    perfumer_map[key] = names

        except Exception:
            continue

print(f"Perfumer data loaded for {len(perfumer_map):,} perfumes")

def get_perfumers(row):
    key = (
        f"{str(row['name']).lower().strip()}|"
        f"{str(row['brand']).lower().strip()}"
    )
    perfumers = perfumer_map.get(key, [])
    return ', '.join(perfumers) if perfumers else None

df['perfumers'] = df.apply(get_perfumers, axis=1)

matched = df['perfumers'].notna().sum()
print(f"Perfumers matched to dataset: {matched:,} perfumes")
print(f"\nSample perfumer entries:")
print(
    df[df['perfumers'].notna()][['name', 'brand', 'perfumers']]
    .head(8)
    .to_string(index=False)
)

Loading perfumer data from perfumes.jsonl...
Perfumer data loaded for 48,890 perfumes
Perfumers matched to dataset: 49,923 perfumes

Sample perfumer entries:
                     name    brand                           perfumers
             Orange Tonic   Azzaro                 Nathalie Feisthauer
                  Amarige Givenchy                    Dominique Ropion
                  Organza Givenchy                        Sophie Labbé
                   Arpège   Lanvin          Andre Fraysse, Paul Vacher
                 Equipage   Hermès                          Guy Robert
Rouge Hermes Eau Delicate   Hermès                         Akiko Kamei
       Eau des Merveilles   Hermès Nathalie Feisthauer, Ralf Schwieger
    Parfum des Merveilles   Hermès  Jean-Claude Ellena, Ralf Schwieger


In [51]:
# Cell 3c — Load Similar Perfumes from perfumes.jsonl
# ─────────────────────────────────────────────────────────────────────
# Similar perfumes live under record['similar']['reminds_me_of']
# Each entry is a dict with a 'slug' field formatted as:
#   'Brand/Perfume-Name-With-Hyphens'
#
# We also extract 'up_votes' and 'down_votes' to filter out
# controversial suggestions — a perfume with 400 up votes and
# 390 down votes is not a reliable similarity signal.
#
# Filtering rule: only keep suggestions where:
#   up_votes > down_votes AND up_votes >= 5
# ─────────────────────────────────────────────────────────────────────

print("Loading similar perfumes from perfumes.jsonl...")
similar_map = {}

def parse_slug(slug):
    """
    Converts 'Giorgio-Armani/Si-Passione-Intense' into
    a readable perfume name 'Si Passione Intense'.
    Brand is before the slash, perfume name after.
    """
    if not slug:
        return None, None
    parts = str(slug).split('/')
    if len(parts) < 2:
        return None, None
    brand_slug  = parts[0].replace('-', ' ').strip()
    perfume_slug = parts[1].replace('-', ' ').strip()
    return perfume_slug, brand_slug

with open('perfumes.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        try:
            record = json.loads(line.strip())
            name   = str(record.get('name',  '')).strip()
            brand  = str(record.get('brand', '')).strip()

            if not name:
                continue

            key     = f"{name.lower()}|{brand.lower()}"
            similar = record.get('similar', {})

            if not similar or not isinstance(similar, dict):
                continue

            reminds_me_of = similar.get('reminds_me_of', [])
            if not reminds_me_of or not isinstance(reminds_me_of, list):
                continue

            valid_similars = []
            for entry in reminds_me_of:
                if not isinstance(entry, dict):
                    continue

                up   = entry.get('up_votes',   0) or 0
                down = entry.get('down_votes',  0) or 0
                slug = entry.get('slug', '')

                # Only keep reliable similarity signals
                if up >= 5 and up > down:
                    perfume_name, perfume_brand = parse_slug(slug)
                    if perfume_name:
                        valid_similars.append({
                            'name':      perfume_name,
                            'brand':     perfume_brand,
                            'up_votes':  up,
                            'down_votes': down,
                            'confidence': round(up / (up + down), 2)
                        })

            if valid_similars:
                # Sort by confidence score, keep top 10
                valid_similars.sort(
                    key=lambda x: x['confidence'], reverse=True
                )
                similar_map[key] = valid_similars[:10]

        except Exception:
            continue

print(f"Similar perfume data loaded for {len(similar_map):,} perfumes")

def get_similar(row):
    key = (
        f"{str(row['name']).lower().strip()}|"
        f"{str(row['brand']).lower().strip()}"
    )
    similar = similar_map.get(key, [])
    if not similar:
        return None
    names = [s['name'] for s in similar if s.get('name')]
    return ', '.join(names) if names else None

df['similar_perfumes'] = df.apply(get_similar, axis=1)

matched = df['similar_perfumes'].notna().sum()
print(f"Similar perfumes matched: {matched:,} perfumes")

# Show quality sample with confidence scores
print(f"\nSample similar perfume entries with confidence:")
sample_keys = list(similar_map.keys())[:3]
for key in sample_keys:
    name_part = key.split('|')[0].title()
    print(f"\n  {name_part}:")
    for s in similar_map[key][:5]:
        print(f"    → {s['name']} "
              f"(↑{s['up_votes']} ↓{s['down_votes']} "
              f"confidence: {s['confidence']})")

Loading similar perfumes from perfumes.jsonl...
Similar perfume data loaded for 41,968 perfumes
Similar perfumes matched: 43,293 perfumes

Sample similar perfume entries with confidence:

  Orange Tonic:
    → Citrus (↑5 ↓4 confidence: 0.56)

  Amarige:
    → Susan (↑328 ↓82 confidence: 0.8)
    → Elysees (↑149 ↓70 confidence: 0.68)
    → Kalanit (↑71 ↓38 confidence: 0.65)
    → Violeta (↑139 ↓83 confidence: 0.63)
    → Blue Lady (↑160 ↓108 confidence: 0.6)

  Organza:
    → Organza Indian Jasmin (↑101 ↓19 confidence: 0.84)
    → Giovany Brezza (↑16 ↓3 confidence: 0.84)
    → Givenchy Harvest 2007 Organza Jasmine (↑99 ↓27 confidence: 0.79)
    → Organza Legere Eau de Toilette (↑33 ↓12 confidence: 0.73)
    → Black (↑319 ↓131 confidence: 0.71)


## Step 6 : Build the TF-IDF Matrix

TF-IDF (Term Frequency - Inverse Document Frequency) converts our
feature strings into numerical vectors that can be compared
mathematically.

### How It Works
- **TF (Term Frequency):** how often a term appears in one perfume's
  feature string. Vanilla appearing 6 times means this perfume is
  heavily vanilla forward.
- **IDF (Inverse Document Frequency):** how rare a term is across ALL
  perfumes. Musk appears in 95,000 perfumes so it gets a lower weight
  than iso-e-super which appears in 200. Rare terms are more
  distinctive and informative.
- **Combined:** TF-IDF rewards terms that are frequent in a specific
  perfume but rare across the dataset — exactly the notes that make
  a perfume unique.

### Parameters Chosen
- `max_features=8000` - top 8,000 most meaningful terms
- `ngram_range=(1,2)` - single words AND pairs like "woody amber"
  or "pink pepper" which are meaningful as units
- `min_df=2` - ignore terms appearing in only one perfume
- `max_df=0.95` - ignore terms in 95%+ of perfumes (too generic)
- `sublinear_tf=True` - dampens effect of very frequent terms so
  vanilla repeated 6 times doesn't completely dominate the vector

In [53]:
# Cell 4 — Build TF-IDF Matrix with Better Discrimination
# ─────────────────────────────────────────────────────────────────────

from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
import scipy.sparse as sp

# Create df_with_features from df
# Only include perfumes that have a feature string
print("Creating df_with_features...")
df_with_features = df[df['feature_string'].notna()].copy()
df_with_features = df_with_features.reset_index(drop=True)
print(f"Perfumes with feature strings: {len(df_with_features):,}")

def build_notes_string(row):
    """Unigram notes only — the core scent ingredients."""
    parts = []
    for col in ['all_notes']:
        val = row.get(col)
        if pd.notna(val) and str(val).strip():
            cleaned = re.sub(r'[|]', ' ', str(val))
            cleaned = re.sub(r'[^\w\s]', ' ', cleaned)
            parts += [cleaned.strip()] * 3
    return ' '.join(parts).lower() if parts else None

def build_accords_string(row):
    """Weighted accords string — already encodes strength via repetition."""
    val = row.get('accords_weighted') or row.get('accords')
    if pd.notna(val) and str(val).strip():
        cleaned = re.sub(r'[^\w\s]', ' ', str(val))
        return cleaned.strip().lower()
    return None

def build_context_string(row):
    """Gender, occasion, season — soft contextual signal."""
    parts = []
    for col in ['gender', 'occasion_tags', 'best_season', 'olfactive_family']:
        val = row.get(col)
        if pd.notna(val) and str(val).strip() not in ['nan', 'Other', '']:
            parts.append(str(val).lower())
    return ' '.join(parts) if parts else 'unisex'

print("Building component strings...")
df_with_features['notes_string']   = df_with_features.apply(
    build_notes_string, axis=1
)
df_with_features['accords_string'] = df_with_features.apply(
    build_accords_string, axis=1
)
df_with_features['context_string'] = df_with_features.apply(
    build_context_string, axis=1
)

notes_strings   = df_with_features['notes_string'].fillna('').tolist()
accords_strings = df_with_features['accords_string'].fillna('').tolist()
context_strings = df_with_features['context_string'].fillna('').tolist()

print(f"Component strings built for {len(df_with_features):,} perfumes")

# ── Vectorizer 1: Notes ────────────────────────────────────────────────
print("\nBuilding notes TF-IDF matrix...")
notes_vectorizer = TfidfVectorizer(
    max_features = 6000,
    ngram_range  = (1, 1),
    min_df       = 2,
    max_df       = 0.98,
    sublinear_tf = True
)
notes_matrix = notes_vectorizer.fit_transform(notes_strings)
print(f"  Notes matrix: {notes_matrix.shape}")

# ── Vectorizer 2: Accords ──────────────────────────────────────────────
print("Building accords TF-IDF matrix...")
accords_vectorizer = TfidfVectorizer(
    max_features = 1500,
    ngram_range  = (1, 2),
    min_df       = 2,
    max_df       = 0.95,
    sublinear_tf = True
)
accords_matrix = accords_vectorizer.fit_transform(accords_strings)
print(f"  Accords matrix: {accords_matrix.shape}")

# ── Vectorizer 3: Context ──────────────────────────────────────────────
print("Building context TF-IDF matrix...")
context_vectorizer = TfidfVectorizer(
    max_features = 500,
    ngram_range  = (1, 1),
    min_df       = 2,
    max_df       = 0.99,
    sublinear_tf = False
)
context_matrix = context_vectorizer.fit_transform(context_strings)
print(f"  Context matrix: {context_matrix.shape}")

# ── Combine ────────────────────────────────────────────────────────────
print("\nCombining matrices with weights...")
tfidf_matrix = hstack([
    notes_matrix   * 0.60,
    accords_matrix * 0.30,
    context_matrix * 0.10
])

print(f"\nFinal TF-IDF matrix:")
print(f"  Shape:   {tfidf_matrix.shape[0]:,} perfumes × "
      f"{tfidf_matrix.shape[1]:,} terms")
print(f"  Memory:  {tfidf_matrix.data.nbytes / 1024 / 1024:.1f} MB")

MIN_FEATURE_LENGTH = 50

# ── Sanity checks ──────────────────────────────────────────────────────
def quick_similarity_check(name_a, name_b, label=''):
    idx_a = df_with_features[
        df_with_features['name'].str.lower() == name_a.lower()
    ].index
    idx_b = df_with_features[
        df_with_features['name'].str.lower() == name_b.lower()
    ].index
    if len(idx_a) == 0 or len(idx_b) == 0:
        print(f"  '{name_a}' ↔ '{name_b}': not found")
        return 0
    sim = cosine_similarity(
        tfidf_matrix[idx_a[0]], tfidf_matrix[idx_b[0]]
    )[0][0]
    tag = f"  [{label}]" if label else ""
    print(f"  '{name_a}' ↔ '{name_b}': {sim:.4f}{tag}")
    return sim

print("\nSanity checks — similar pairs should score higher:")
quick_similarity_check("Aventus", "Al Dur Al Maknoon Silver",
                        "known Aventus dupe")
quick_similarity_check("Aventus", "Aventus for Her",
                        "flanker")
quick_similarity_check("Aventus", "Silver Mountain Water",
                        "shares notes")

print("\nSanity checks — different pairs should score lower:")
quick_similarity_check("Aventus", "Amarige",   "different family")
quick_similarity_check("Aventus", "Angel",     "different family")

print("\nCell 4 complete!")

Creating df_with_features...
Perfumes with feature strings: 150,288
Building component strings...
Component strings built for 150,288 perfumes

Building notes TF-IDF matrix...
  Notes matrix: (150288, 1705)
Building accords TF-IDF matrix...
  Accords matrix: (150288, 1500)
Building context TF-IDF matrix...
  Context matrix: (150288, 3)

Combining matrices with weights...

Final TF-IDF matrix:
  Shape:   150,288 perfumes × 3,208 terms
  Memory:  28.2 MB

Sanity checks — similar pairs should score higher:
  'Aventus' ↔ 'Al Dur Al Maknoon Silver': 0.4179  [known Aventus dupe]
  'Aventus' ↔ 'Aventus for Her': 0.3245  [flanker]
  'Aventus' ↔ 'Silver Mountain Water': 0.1861  [shares notes]

Sanity checks — different pairs should score lower:
  'Aventus' ↔ 'Amarige': 0.1004  [different family]
  'Aventus' ↔ 'Angel': 0.1109  [different family]

Cell 4 complete!


## Step 7 : Dupe Detection

Now that the TF-IDF matrix exists we can compute similarity between
every budget perfume and every luxury perfume at scale.

### What Is a Dupe?
A dupe (duplicate) is a budget-friendly perfume that smells
significantly similar to an expensive original. The fragrance
community has built an entire culture around dupes - finding
a 25 dollar alternative that captures 85% of a $400 perfume's DNA.

### Our Approach
We compute cosine similarity between every budget/mid tier perfume
and every luxury/premium perfume using the TF-IDF vectors. If the
similarity exceeds 0.75, we flag the budget perfume as a potential
dupe and record which luxury perfume it most resembles.

This powers the "Smell rich, not broke" feature in the app - one
of Beyond Fragrancy's most distinctive offerings.

### Why 0.75 Threshold?
Too high (0.90+) and we miss genuine dupes that differ slightly in
one note family. Too low (0.60) and we flag perfumes that are merely
in the same broad family. 0.75 catches meaningful similarity while
filtering out generic category overlap.

In [54]:
# Cell 4b : Dupe Detection
# ─────────────────────────────────────────────────────────────────────
# Fix: exclude perfumes with sparse feature strings (fewer than 50
# characters) from being dupe targets. A perfume with only 'male' or
# 'unisex' as its feature string cannot meaningfully represent a scent
# and produces false 1.0 similarity matches against anything.
# ─────────────────────────────────────────────────────────────────────

DUPE_THRESHOLD       = 0.75
MIN_FEATURE_LENGTH   = 50  # minimum characters for a meaningful feature string

# Flag perfumes with meaningful feature strings
df_with_features['feature_length'] = df_with_features[
    'feature_string'
].str.len().fillna(0)

meaningful_mask = df_with_features['feature_length'] >= MIN_FEATURE_LENGTH

print(f"Perfumes with meaningful feature strings: "
      f"{meaningful_mask.sum():,} / {len(df_with_features):,}")
print(f"Excluded (too sparse): "
      f"{(~meaningful_mask).sum():,}")

# Luxury/premium originals WITH meaningful feature strings only
luxury_mask = (
    df_with_features['price_tier'].isin(['luxury', 'premium']) &
    (df_with_features['flanker_type'] == 'original') &
    meaningful_mask
)

# Budget/mid perfumes WITH meaningful feature strings only
budget_mask = (
    df_with_features['price_tier'].isin(['budget', 'mid']) &
    meaningful_mask
)

luxury_indices = df_with_features[luxury_mask].index.tolist()
budget_indices = df_with_features[budget_mask].index.tolist()

print(f"\nLuxury/premium originals (meaningful): {len(luxury_indices):,}")
print(f"Budget/mid perfumes (meaningful):      {len(budget_indices):,}")
print(f"Dupe threshold: {DUPE_THRESHOLD}")
print(f"Running dupe detection...")

luxury_matrix = tfidf_matrix[luxury_indices]
dupe_results  = {}
chunk_size    = 2000

for i in range(0, len(budget_indices), chunk_size):
    chunk_indices = budget_indices[i:i + chunk_size]
    chunk_matrix  = tfidf_matrix[chunk_indices]
    similarities  = cosine_similarity(chunk_matrix, luxury_matrix)

    for j, sim_row in enumerate(similarities):
        best_idx = sim_row.argmax()
        best_sim = sim_row[best_idx]

        if best_sim >= DUPE_THRESHOLD:
            budget_df_idx = chunk_indices[j]
            luxury_df_idx = luxury_indices[best_idx]
            budget_name   = df_with_features.loc[budget_df_idx, 'name']
            luxury_name   = df_with_features.loc[luxury_df_idx, 'name']
            luxury_brand  = df_with_features.loc[luxury_df_idx, 'brand']
            luxury_tier   = df_with_features.loc[luxury_df_idx, 'price_tier']

            dupe_results[budget_name] = {
                'dupe_of':          luxury_name,
                'dupe_brand':       luxury_brand,
                'dupe_tier':        luxury_tier,
                'dupe_similarity':  round(float(best_sim), 3)
            }

    if (i // chunk_size) % 5 == 0:
        print(f"  Processed {min(i + chunk_size, len(budget_indices)):,}"
              f"/{len(budget_indices):,} budget perfumes...")

print(f"\nDupe detection complete!")
print(f"Dupes identified: {len(dupe_results):,}")

# Apply back to dataframes
for target_df in [df_with_features, df]:
    target_df['dupe_of'] = target_df['name'].map(
        lambda n: dupe_results.get(n, {}).get('dupe_of')
    )
    target_df['dupe_brand'] = target_df['name'].map(
        lambda n: dupe_results.get(n, {}).get('dupe_brand')
    )
    target_df['dupe_similarity'] = target_df['name'].map(
        lambda n: dupe_results.get(n, {}).get('dupe_similarity')
    )

# Show clean sample dupes — exclude perfect 1.0 scores
# to see genuinely interesting matches
clean_dupes = df_with_features[
    df_with_features['dupe_of'].notna() &
    (df_with_features['dupe_similarity'] < 0.999)
].nlargest(15, 'dupe_similarity')[
    ['name', 'brand', 'price_tier',
     'dupe_of', 'dupe_brand', 'dupe_similarity']
]

print(f"\nTop 15 most interesting dupes (similarity < 1.0):")
print(clean_dupes.to_string(index=False))

# Distribution
print(f"\nDupe similarity distribution:")
sim_data = df_with_features['dupe_similarity'].dropna()
print(f"  Total dupes:     {len(sim_data):,}")
print(f"  Perfect (1.0):   {(sim_data == 1.0).sum():,}")
print(f"  High (0.9-0.99): {((sim_data >= 0.9) & (sim_data < 1.0)).sum():,}")
print(f"  Good (0.75-0.9): {((sim_data >= 0.75) & (sim_data < 0.9)).sum():,}")
print(f"  Mean similarity: {sim_data.mean():.3f}")

# Sanity check similarities with correct names
print(f"\nSanity checks with correct perfume names:")
def quick_similarity_check(name_a, name_b):
    idx_a = df_with_features[
        df_with_features['name'].str.lower() == name_a.lower()
    ].index
    idx_b = df_with_features[
        df_with_features['name'].str.lower() == name_b.lower()
    ].index
    if len(idx_a) == 0 or len(idx_b) == 0:
        print(f"  '{name_a}' ↔ '{name_b}': not found")
        return
    sim = cosine_similarity(
        tfidf_matrix[idx_a[0]], tfidf_matrix[idx_b[0]]
    )[0][0]
    print(f"  '{name_a}' ↔ '{name_b}': {sim:.4f}")

print("Similar pairs (should score higher):")
quick_similarity_check("Aventus",       "Silver Mountain Water")
quick_similarity_check("Eau Sauvage",   "Neroli Sauvage")
quick_similarity_check("Aventus",       "Aventus for Her")

print("Different pairs (should score lower):")
quick_similarity_check("Aventus",       "Amarige")
quick_similarity_check("Eau Sauvage",   "Amarige")
quick_similarity_check("Aventus",       "Angel")

Perfumes with meaningful feature strings: 146,910 / 150,288
Excluded (too sparse): 3,378

Luxury/premium originals (meaningful): 6,595
Budget/mid perfumes (meaningful):      139,422
Dupe threshold: 0.75
Running dupe detection...
  Processed 2,000/139,422 budget perfumes...
  Processed 12,000/139,422 budget perfumes...
  Processed 22,000/139,422 budget perfumes...
  Processed 32,000/139,422 budget perfumes...
  Processed 42,000/139,422 budget perfumes...
  Processed 52,000/139,422 budget perfumes...
  Processed 62,000/139,422 budget perfumes...
  Processed 72,000/139,422 budget perfumes...
  Processed 82,000/139,422 budget perfumes...
  Processed 92,000/139,422 budget perfumes...
  Processed 102,000/139,422 budget perfumes...
  Processed 112,000/139,422 budget perfumes...
  Processed 122,000/139,422 budget perfumes...
  Processed 132,000/139,422 budget perfumes...

Dupe detection complete!
Dupes identified: 7,366

Top 15 most interesting dupes (similarity < 1.0):
                       

## Step 8 : The Recommendation Engine

This is the core of Beyond Fragrancy. The recommend function combines
five signals into a single ranked list of perfumes:

1. **TF-IDF cosine similarity** - how closely the scent DNA matches
2. **Similar perfumes from Fragrantica** - behavioral signal from
   real user browsing patterns (proxy collaborative filtering)
3. **Popularity weighting** - confidence-adjusted rating score
4. **Perfumer affinity** - bonus for perfumes by the same nose
5. **Diversity enforcement** - ensures variety across scent families

### Two Input Modes
- **By perfume name** - user tells us what they already love
- **By notes** - user describes what they want directly

### Filters Available
Budget tier, gender, occasion, season - all optional and gracefully
degraded if they produce too few results.

In [60]:
# Cell 5 — Complete Recommendation Engine
# ─────────────────────────────────────────────────────────────────────

from sklearn.preprocessing import MinMaxScaler
from rapidfuzz import fuzz, process
from datetime import date
import scipy.sparse as sp

# ── Helper functions ───────────────────────────────────────────────────

def get_current_season():
    month = date.today().month
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else:                     return 'Autumn'

def average_sparse_rows(matrix, indices):
    """
    Safely averages sparse matrix rows.
    Returns a (1, n_features) sparse matrix.
    """
    stacked  = sp.vstack([matrix[i] for i in indices])
    averaged = stacked.mean(axis=0)
    return sp.csr_matrix(np.asarray(averaged))

def find_perfume_index(name, df_source):
    """
    Finds perfume by name — exact match first, fuzzy fallback.
    Returns (index, matched_name, match_score) or (None, None, 0).
    """
    name_lower = name.lower().strip()
    exact = df_source[
        df_source['name'].str.lower().str.strip() == name_lower
    ]
    if len(exact) > 0:
        idx = exact.index[0]
        return idx, df_source.loc[idx, 'name'], 100

    match = process.extractOne(
        name, df_source['name'].tolist(), scorer=fuzz.ratio
    )
    if match and match[1] >= 70:
        matched_name = match[0]
        idx = df_source[
            df_source['name'] == matched_name
        ].index[0]
        return idx, matched_name, match[1]

    return None, None, 0

def boost_diversity(results_df, n_results):
    """
    Ensures recommendations span multiple olfactive families.
    No more than 40% of results from any single family.
    """
    max_per_family = max(2, int(n_results * 0.4))
    family_counts  = {}
    selected       = []

    for _, row in results_df.iterrows():
        if len(selected) >= n_results:
            break
        family = str(row.get('olfactive_family', 'Other'))
        count  = family_counts.get(family, 0)
        if count < max_per_family:
            selected.append(row)
            family_counts[family] = count + 1

    if len(selected) < n_results:
        selected_names = {r['name'] for r in selected}
        remaining = results_df[
            ~results_df['name'].isin(selected_names)
        ].head(n_results - len(selected))
        for _, row in remaining.iterrows():
            selected.append(row)

    return pd.DataFrame(selected)

def get_perfumer_boost(results_df, seed_perfumers):
    """
    Adds up to 0.15 to final_score for shared perfumers.
    """
    if not seed_perfumers or 'perfumers' not in results_df.columns:
        return results_df

    def perfumer_score(row_perfumers):
        if pd.isna(row_perfumers):
            return 0
        row_p   = set(str(row_perfumers).lower().split(', '))
        matches = row_p.intersection(seed_perfumers)
        return min(len(matches) * 0.05, 0.15)

    results_df = results_df.copy()
    results_df['perfumer_boost'] = results_df['perfumers'].apply(
        perfumer_score
    )
    results_df['final_score'] = (
        results_df['final_score'] + results_df['perfumer_boost']
    ).clip(upper=1.0)
    return results_df

def make_smart_dedup_key(row):
    """
    Creates dedup key handling variants like:
    'Kalanit' and 'Kalanit O Boticário' — same perfume, different entries.
    """
    fg    = str(row.get('flanker_group', '')).strip()
    name  = str(row.get('name',  '')).lower().strip()
    brand = str(row.get('brand', '')).lower().strip()

    if fg and fg.lower() not in ['nan', '', 'none']:
        return fg.lower()

    brand_words = [w for w in brand.split() if len(w) > 2]
    name_clean  = name
    for word in brand_words:
        name_clean = name_clean.replace(word, '').strip()

    words = [w for w in name_clean.split() if len(w) > 2][:3]
    return f"{' '.join(words)}|{brand}"

# ── Main recommend function ────────────────────────────────────────────

def recommend(
    perfume_names = None,
    notes_input   = None,
    budget_tier   = None,
    gender        = None,
    occasion      = None,
    season        = 'auto',
    exclude_owned = True,
    n_results     = 10,
    diversity     = True,
    show_dupes    = False
):
    """
    Core Beyond Fragrancy recommendation function.

    Parameters
    ----------
    perfume_names : list of str
        Perfumes the user loves.
    notes_input : str
        Free text description of desired notes.
    budget_tier : str or None
        'budget', 'mid', 'premium', 'luxury' or None for any.
    gender : str or None
        'male', 'female', 'unisex' or None for any.
    occasion : str or None
        'office', 'date', 'casual', 'evening', 'wedding'.
    season : str
        'Winter', 'Spring', 'Summer', 'Autumn' or 'auto'.
    exclude_owned : bool
        Exclude seed perfumes from results.
    n_results : int
        Number of recommendations to return.
    diversity : bool
        Enforce olfactive family diversity.
    show_dupes : bool
        If True, filter to budget/mid alternatives only.
    """

    if perfume_names is None and notes_input is None:
        print("Please provide perfume names or note descriptions.")
        return None

    seed_perfumers    = set()
    similar_name_pool = []
    query_vector      = None

    # ── MODE 1: By perfume name ────────────────────────────────────────
    if perfume_names:
        seed_indices = []

        for name in perfume_names:
            idx, matched, score = find_perfume_index(
                name, df_with_features
            )
            if idx is None:
                print(f"Could not find '{name}' — skipping")
                continue
            if score == 100:
                print(f"Found: '{matched}' by "
                      f"{df_with_features.loc[idx, 'brand']}")
            else:
                print(f"Fuzzy match: '{name}' → '{matched}' "
                      f"by {df_with_features.loc[idx, 'brand']} "
                      f"({score}% match)")

            seed_indices.append(idx)

            p = df_with_features.loc[idx, 'perfumers']
            if pd.notna(p):
                for perf in str(p).split(', '):
                    seed_perfumers.add(perf.lower().strip())

            sim = df_with_features.loc[idx, 'similar_perfumes']
            if pd.notna(sim):
                similar_name_pool.extend(str(sim).split(', ')[:5])

        if not seed_indices:
            print("None of the provided perfumes were found.")
            return None

        query_vector = average_sparse_rows(tfidf_matrix, seed_indices)

        similar_indices = []
        for sim_name in similar_name_pool:
            idx_s, _, score_s = find_perfume_index(
                sim_name.strip(), df_with_features
            )
            if idx_s is not None and score_s >= 80:
                similar_indices.append(idx_s)

        if similar_indices:
            sim_vec      = average_sparse_rows(tfidf_matrix, similar_indices)
            query_vector = (
                query_vector.multiply(0.80) + sim_vec.multiply(0.20)
            )

    # ── MODE 2: By notes description ──────────────────────────────────
    else:
        cleaned    = re.sub(r'[^\w\s]', ' ', notes_input.lower())
        notes_part = notes_vectorizer.transform([cleaned]) * 0.60
        empty_acc  = sp.csr_matrix((1, accords_matrix.shape[1]))
        empty_ctx  = sp.csr_matrix((1, context_matrix.shape[1]))
        query_vector = hstack([notes_part, empty_acc, empty_ctx])

    # ── SIMILARITY SEARCH ──────────────────────────────────────────────
    similarity_scores = cosine_similarity(
        query_vector, tfidf_matrix
    ).flatten()

    results = df_with_features.copy()
    results['similarity_score'] = similarity_scores

    if exclude_owned and perfume_names:
        owned_lower = [n.lower().strip() for n in perfume_names]
        results = results[
            ~results['name'].str.lower().str.strip().isin(owned_lower)
        ]

    results = results[results['feature_length'] >= MIN_FEATURE_LENGTH]

    # ── FILTERS ────────────────────────────────────────────────────────
    if budget_tier:
        filtered = results[results['price_tier'] == budget_tier]
        if len(filtered) >= 5:
            results = filtered
        else:
            print(f"Only {len(filtered)} results for '{budget_tier}' "
                  f"— showing all budgets")

    if gender and gender != 'unisex':
        g_filtered = results[results['gender'].isin([gender, 'unisex'])]
        if len(g_filtered) >= 5:
            results = g_filtered

    if occasion:
        if 'occasion_tags' in results.columns:
            occ_filtered = results[
                results['occasion_tags'].str.contains(
                    occasion, case=False, na=False
                )
            ]
            if len(occ_filtered) >= 5:
                results = occ_filtered

    if season == 'auto':
        season = get_current_season()
        print(f"Season: {season} (auto-detected)")

    if season and 'best_season' in results.columns:
        sea_filtered = results[results['best_season'] == season]
        if len(sea_filtered) >= 5:
            results = sea_filtered

    # ── SCORING ────────────────────────────────────────────────────────
    results = results.copy()
    scaler  = MinMaxScaler()

    if len(results) > 1:
        results['sim_norm'] = scaler.fit_transform(
            results[['similarity_score']]
        )
        if 'popularity_score' in results.columns:
            results['pop_norm'] = scaler.fit_transform(
                results[['popularity_score']].fillna(0)
            )
        else:
            results['pop_norm'] = 0
    else:
        results['sim_norm'] = results['similarity_score']
        results['pop_norm'] = 0

    # 60% similarity + 40% popularity
    results['final_score'] = (
        0.60 * results['sim_norm'] +
        0.40 * results['pop_norm']
    )

    if seed_perfumers:
        results = get_perfumer_boost(results, seed_perfumers)
        boosted = (
            results.get('perfumer_boost', pd.Series([0])) > 0
        ).sum()
        if boosted > 0:
            print(f"Perfumer affinity boost: {boosted} perfumes")

    # Multi-level sort to break ties
    results['rating_avg_fill']   = results['rating_avg'].fillna(0)
    results['rating_count_fill'] = results['rating_count'].fillna(0)
    results = results.sort_values(
        ['final_score', 'rating_avg_fill', 'rating_count_fill'],
        ascending=[False, False, False]
    )

    # Smart deduplication
    results['dedup_key'] = results.apply(make_smart_dedup_key, axis=1)
    results = results.drop_duplicates(subset=['dedup_key'], keep='first')

    if diversity and len(results) >= n_results:
        results = boost_diversity(results, n_results * 3)

    if show_dupes:
        results = results[results['price_tier'].isin(['budget', 'mid'])]

    top = results.head(n_results)

    output_cols = [
        'name', 'brand', 'gender', 'price_tier', 'olfactive_family',
        'best_season', 'accords', 'rating_avg', 'rating_count',
        'similarity_score', 'final_score', 'confidence_score',
        'dupe_of', 'dupe_similarity', 'parent_perfume', 'flanker_type'
    ]
    if 'perfumers' in top.columns:
        output_cols.insert(3, 'perfumers')

    return top[[c for c in output_cols if c in top.columns]]

print("Recommendation engine ready!")
print("All helper functions defined:")
print("  find_perfume_index, average_sparse_rows, boost_diversity")
print("  get_perfumer_boost, make_smart_dedup_key, recommend")

Recommendation engine ready!
All helper functions defined:
  find_perfume_index, average_sparse_rows, boost_diversity
  get_perfumer_boost, make_smart_dedup_key, recommend


In [61]:
# Cell 5b — Compute missing columns and fix deduplication

# ── Fix 1: Recompute popularity_score directly on df_with_features ────

print("Computing popularity score...")

def bayesian_popularity(rating_avg, rating_count, confidence_weight=1.0):
    """
    Bayesian average blends actual rating with a prior of 3.5
    based on 50 phantom votes. More votes = more trust in real rating.
    Multiplied by confidence_weight to dampen low-vote perfumes.
    """
    if pd.isna(rating_avg) or pd.isna(rating_count):
        return 0.0
    prior        = 3.5
    prior_weight = 50
    bayesian     = (
        (prior * prior_weight) + (rating_avg * rating_count)
    ) / (prior_weight + rating_count)
    return round(bayesian * confidence_weight, 4)

# Compute confidence weight from rating_count
def get_confidence_weight(rating_count):
    if pd.isna(rating_count) or rating_count == 0:
        return 0.2
    elif rating_count < 50:
        return 0.2
    elif rating_count < 500:
        return 0.5
    elif rating_count < 5000:
        return 0.8
    else:
        return 1.0

df_with_features['confidence_weight'] = df_with_features[
    'rating_count'
].apply(get_confidence_weight)

df_with_features['popularity_score'] = df_with_features.apply(
    lambda r: bayesian_popularity(
        r['rating_avg'],
        r['rating_count'],
        r['confidence_weight']
    ), axis=1
)

print(f"Popularity score computed!")
print(f"\nPopularity score stats:")
print(df_with_features['popularity_score'].describe().round(3))
print(f"\nTop 10 perfumes by popularity score:")
print(
    df_with_features[df_with_features['popularity_score'] > 0]
    .nlargest(10, 'popularity_score')
    [['name', 'brand', 'rating_avg', 'rating_count', 'popularity_score']]
    .to_string(index=False)
)

# Also transfer olfactive_family, best_season, occasion_tags
# from df if they exist there but not in df_with_features
print("\nChecking for missing enrichment columns...")
enrichment_cols = [
    'olfactive_family', 'best_season', 'occasion_tags',
    'note_complexity', 'limited_edition', 'days_since_release',
    'dupe_of', 'dupe_brand', 'dupe_similarity'
]

for col in enrichment_cols:
    if col not in df_with_features.columns:
        if col in df.columns:
            # Match by position since both dfs came from same source
            df_with_features[col] = df[col].values[:len(df_with_features)]
            print(f"  Transferred: {col}")
        else:
            print(f"  Missing from both: {col}")
    else:
        print(f"  Already present: {col}")

# ── Fix 2: Smart deduplication key ────────────────────────────────────
print("\nBuilding smart deduplication keys...")

def make_smart_dedup_key(row):
    """
    Creates a dedup key that handles cases like:
    'Kalanit' and 'Kalanit O Boticário' — same perfume, different entries
    Uses flanker_group as primary key, falls back to normalised name.
    """
    fg    = str(row.get('flanker_group', '')).strip()
    name  = str(row.get('name',  '')).lower().strip()
    brand = str(row.get('brand', '')).lower().strip()

    if fg and fg.lower() not in ['nan', '', 'none']:
        return fg.lower()

    # Remove brand words from perfume name
    brand_words = [w for w in brand.split() if len(w) > 2]
    name_clean  = name
    for word in brand_words:
        name_clean = name_clean.replace(word, '').strip()

    # Take first 3 meaningful words
    words = [w for w in name_clean.split() if len(w) > 2][:3]
    return f"{' '.join(words)}|{brand}"

df_with_features['smart_dedup_key'] = df_with_features.apply(
    make_smart_dedup_key, axis=1
)

# Verify on Kalanit
kalanit = df_with_features[
    df_with_features['name'].str.contains('Kalanit', na=False)
][['name', 'brand', 'flanker_group', 'smart_dedup_key']].head(5)
print(f"\nKalanit dedup check:")
print(kalanit.to_string(index=False))

# Verify on BR540 dupes
br540 = df_with_features[
    df_with_features['name'].str.contains(
        'Rouge Obsession|Aquila|Haan Amber|Ruby Whispers',
        na=False
    )
][['name', 'brand', 'smart_dedup_key', 'popularity_score']].head(8)
print(f"\nBR540 dupe dedup check:")
print(br540.to_string(index=False))

Computing popularity score...
Popularity score computed!

Popularity score stats:
count    150288.000
mean          1.121
std           0.737
min           0.000
25%           0.702
50%           0.718
75%           1.816
max           4.599
Name: popularity_score, dtype: float64

Top 10 perfumes by popularity score:
                                      name              brand  rating_avg  rating_count  popularity_score
                    The Most Wanted Parfum             Azzaro      4.6045       10522.0            4.5993
                         Le Male Le Parfum Jean Paul Gaultier      4.5933       26470.0            4.5912
Emporio Armani Stronger With You Intensely     Giorgio Armani      4.5844       23874.0            4.5821
                           The Most Wanted             Azzaro      4.5752       11102.0            4.5704
 Valentino Uomo Born In Roma Coral Fantasy          Valentino      4.5747        8770.0            4.5686
                               Imagination   

In [62]:
# Save checkpoint so Colab restarts don't lose processed data
print("Saving processed checkpoint...")
df_with_features.to_pickle('df_with_features_checkpoint.pkl')
print("Saved df_with_features_checkpoint.pkl")

# Also save the key variables we'll need
import pickle
with open('vectorizers_checkpoint.pkl', 'wb') as f:
    pickle.dump({
        'notes_vectorizer':   notes_vectorizer,
        'accords_vectorizer': accords_vectorizer,
        'context_vectorizer': context_vectorizer,
        'autocorrect_map':    autocorrect_map,
        'similar_map':        similar_map,
        'dupe_results':       dupe_results
    }, f)
print("Saved vectorizers_checkpoint.pkl")

import scipy.sparse as sp
sp.save_npz('tfidf_matrix_checkpoint.npz', tfidf_matrix.tocsr())
print("Saved tfidf_matrix_checkpoint.npz")

Saving processed checkpoint...
Saved df_with_features_checkpoint.pkl
Saved vectorizers_checkpoint.pkl
Saved tfidf_matrix_checkpoint.npz


In [63]:
# Cell 6 — Test the Recommendation Engine
# ─────────────────────────────────────────────────────────────────────
# Four tests covering the main use cases:
#   Test 1 — Single perfume input
#   Test 2 — Notes description input
#   Test 3 — Multiple perfumes + filters
#   Test 4 — Dupe finder mode
# ─────────────────────────────────────────────────────────────────────

def print_results(results, title):
    print(f"\n{'='*65}")
    print(f" {title}")
    print(f"{'='*65}")
    if results is None or len(results) == 0:
        print("No results found.")
        return
    display_cols = [
        'name', 'brand', 'price_tier', 'olfactive_family',
        'similarity_score', 'final_score'
    ]
    display_cols = [c for c in display_cols if c in results.columns]
    print(results[display_cols].to_string(index=False))

# Test 1 — Single perfume
results1 = recommend(
    perfume_names = ["Aventus"],
    n_results     = 10
)
print_results(results1, "TEST 1: Similar to Aventus by Creed")

# Test 2 — Notes input
results2 = recommend(
    notes_input  = "vanilla oud amber warm spicy woody",
    budget_tier  = "budget",
    n_results    = 10
)
print_results(results2, "TEST 2: Vanilla oud amber — budget tier")

# Test 3 — Multiple perfumes + filters
results3 = recommend(
    perfume_names = ["Amarige", "Organza"],
    gender        = "female",
    budget_tier   = "mid",
    n_results     = 10
)
print_results(results3,
    "TEST 3: Like Amarige + Organza — female, mid budget")

# Test 4 — Dupe finder
results4 = recommend(
    perfume_names = ["Baccarat Rouge 540"],
    show_dupes    = True,
    n_results     = 10
)
print_results(results4,
    "TEST 4: Affordable dupes of Baccarat Rouge 540")

Found: 'Aventus' by Creed
Season: Summer (auto-detected)
Perfumer affinity boost: 125 perfumes

 TEST 1: Similar to Aventus by Creed
                                    name           brand price_tier  similarity_score  final_score
                Club de Nuit Intense Man           Armaf     budget          0.610094     0.771622
Supremacy Collector's Edition Pour Homme           Afnan     budget          0.525121     0.749651
        Club De Nuit Intense Man Extrait           Armaf     budget          0.638406     0.735542
                                     Now            RAVE     budget          0.643757     0.728055
                        Supremacy Silver           Afnan     budget          0.647426     0.723220
                               Insidious Insider Parfums        mid          0.738026     0.676508
                 Club de Nuit Precieux I           Armaf     budget          0.534503     0.675270
                                 Veritas            Lidl        mid        

In [64]:
# Cell 7 — Save all checkpoints
# ─────────────────────────────────────────────────────────────────────
# Saves the processed dataframe, TF-IDF matrix, and all fitted
# vectorizers so a Colab restart only needs Cell 0 to reload.
# ─────────────────────────────────────────────────────────────────────

import pickle
import scipy.sparse as sp

print("Saving checkpoints...")

# Save processed dataframe
df_with_features.to_pickle('df_with_features_checkpoint.pkl')
print(f"  df_with_features saved: {len(df_with_features):,} perfumes")

# Save TF-IDF matrix
sp.save_npz('tfidf_matrix_checkpoint.npz', tfidf_matrix.tocsr())
print(f"  tfidf_matrix saved: {tfidf_matrix.shape}")

# Save all fitted objects
checkpoint = {
    'notes_vectorizer':   notes_vectorizer,
    'accords_vectorizer': accords_vectorizer,
    'context_vectorizer': context_vectorizer,
    'autocorrect_map':    autocorrect_map,
    'similar_map':        similar_map,
    'dupe_results':       dupe_results,
}
with open('checkpoints.pkl', 'wb') as f:
    pickle.dump(checkpoint, f)
print("  vectorizers and maps saved")

# Download all three to your local machine
from google.colab import files
files.download('df_with_features_checkpoint.pkl')
files.download('tfidf_matrix_checkpoint.npz')
files.download('checkpoints.pkl')

print("\nAll checkpoints saved and downloading!")
print("Store these in Beyond-Fragrancy/models/ folder")

Saving checkpoints...
  df_with_features saved: 150,288 perfumes
  tfidf_matrix saved: (150288, 3208)
  vectorizers and maps saved


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


All checkpoints saved and downloading!
Store these in Beyond-Fragrancy/models/ folder
